# DBMF sizing

Research notebook for sizing a managed-futures sleeve against a self-financing baseline portfolio of `80% equity + 100% 10Y bond proxy + 10% commodity proxy - 90% cash proxy`.

The key question is not standalone DBMF Sharpe. It is whether a DBMF / managed-futures proxy improves marginal risk contribution, monthly CVaR, max drawdown, and equity-bond selloff behavior.

Data notes:
- Managed-futures history comes from the cleaned repo CSV at `data/interim/sg_managed_futures_indices.csv`.
- `SG_Trend_Index` is the default long-history DBMF proxy; switch `MF_INDEX_COLUMN` to `SG_CTA_Index` to test the broader CTA index.
- The managed-futures proxy uses DBMF total returns when DBMF live history exists, and SG Trend total returns before DBMF inception.
- Vol targeting is applied to excess returns over DTB3 cash, then cash daily return is added back. Current assumptions: DBMF excess vol target = 10%; SG Trend excess vol target = 12%.
- Cash total return is built from FRED `DTB3`: convert the T-bill bank discount rate into a daily investment rate, then accrue across calendar-day gaps.

In [ ]:
import numpy as np
import pandas as pd

from alpha_lab.analytics.returns import cumulative_returns, drawdown
from alpha_lab.analytics.risk import cov_matrix, cvar, risk_contributions
from alpha_lab.backtest.metrics import summary
from alpha_lab.data.loaders.fred import load_cash_total_return_index
from alpha_lab.data.loaders.yfinance import load_prices
from alpha_lab.portfolio.long_only import fixed_weight_returns
from alpha_lab.reporting.charts import drawdown_chart, equity_curve, heatmap_monthly
from alpha_lab.utils.paths import DATA_DIR

In [ ]:
START = "2000-01-01"
END = None

# ETF proxies keep the research reproducible with public Yahoo data. Replace IEF/DBC with
# local futures total-return series if available.
EQUITY_TICKER = "SPY"
BOND_TICKER = "IEF"
COMMODITY_TICKER = "DBC"
DBMF_TICKER = "DBMF"
CASH_TICKER = "CASH"
SG_INDEX_PATH = DATA_DIR / "interim" / "sg_managed_futures_indices.csv"
MF_INDEX_COLUMN = "SG_Trend_Index"

CORE_WEIGHTS = {
    EQUITY_TICKER: 0.80,
    BOND_TICKER: 1.00,
    COMMODITY_TICKER: 0.10,
    CASH_TICKER: -0.90,
}

DBMF_WEIGHTS = [0.00, 0.05, 0.075, 0.10, 0.125, 0.15, 0.20]

# Fund DBMF from cash until cash reaches this floor, then use secondary funding.
CASH_WEIGHT_FLOOR = -1.00
PRIMARY_FUNDING_WEIGHTS = {
    CASH_TICKER: 1.00,
}
SECONDARY_FUNDING_WEIGHTS = {
    EQUITY_TICKER: 1.00,
}

DBMF_EXCESS_TARGET_VOL = 0.12
SG_EXCESS_TARGET_VOL = 0.11
MONTHLY_CVAR_Q = 0.05
ROLLING_WINDOW_MONTHS = 36
MIN_STRESS_MONTHS = 6

## SG Trend / CTA data

The committed CSV stores daily index levels for both SG Trend and SG CTA. The notebook converts the selected index level and DBMF adjusted close into daily total returns, subtracts DTB3 cash return for vol scaling, then adds cash return back.

In [ ]:
sg_indices = pd.read_csv(SG_INDEX_PATH, parse_dates=["date"]).set_index("date").sort_index()
sg_returns = sg_indices.pct_change().dropna(how="all")
sg_indices.tail()

In [ ]:
tickers = [EQUITY_TICKER, BOND_TICKER, COMMODITY_TICKER, DBMF_TICKER]
prices = load_prices(tickers, start=START, end=END).dropna(how="all")
cash_index = load_cash_total_return_index(start=START, end=END).rename(CASH_TICKER)

daily_returns = prices.pct_change(fill_method=None).dropna(subset=[EQUITY_TICKER, BOND_TICKER, COMMODITY_TICKER], how="any")
cash_index_aligned = cash_index.reindex(cash_index.index.union(prices.index).union(sg_indices.index)).ffill()
cash_returns_full = cash_index_aligned.pct_change(fill_method=None).fillna(0.0)
daily_returns[CASH_TICKER] = cash_returns_full.reindex(daily_returns.index).fillna(0.0)
prices.tail()

In [ ]:
sg_total_returns = sg_returns[MF_INDEX_COLUMN].dropna().rename(f"{MF_INDEX_COLUMN} total")
dbmf_total_returns = daily_returns[DBMF_TICKER].dropna().rename("DBMF total")
dbmf_start = dbmf_total_returns.index.min()
source_label = f"{MF_INDEX_COLUMN} before {dbmf_start.date()}, DBMF from {dbmf_start.date()}"

cash_for_sg = cash_returns_full.reindex(sg_total_returns.index).fillna(0.0)
cash_for_dbmf = cash_returns_full.reindex(dbmf_total_returns.index).fillna(0.0)

sg_excess_returns = (sg_total_returns - cash_for_sg).rename(f"{MF_INDEX_COLUMN} excess")
dbmf_excess_returns = (dbmf_total_returns - cash_for_dbmf).rename("DBMF excess")

sg_realized_excess_vol = sg_excess_returns.std() * np.sqrt(252)
dbmf_realized_excess_vol = dbmf_excess_returns.std() * np.sqrt(252)
sg_scaled_excess_returns = sg_excess_returns * (SG_EXCESS_TARGET_VOL / sg_realized_excess_vol)
dbmf_scaled_excess_returns = dbmf_excess_returns * (DBMF_EXCESS_TARGET_VOL / dbmf_realized_excess_vol)

sg_scaled_total_returns = (sg_scaled_excess_returns + cash_for_sg).rename("SG proxy total")
dbmf_scaled_total_returns = (dbmf_scaled_excess_returns + cash_for_dbmf).rename("DBMF proxy total")
mf_returns = sg_scaled_total_returns.copy().rename("Managed futures DBMF/SG proxy")
mf_returns.loc[dbmf_start:] = dbmf_scaled_total_returns.reindex(mf_returns.loc[dbmf_start:].index)
mf_returns = mf_returns.dropna()

pd.Series(
    {
        "sg_source": MF_INDEX_COLUMN,
        "dbmf_start": dbmf_start,
        "proxy_start": mf_returns.index.min(),
        "proxy_end": mf_returns.index.max(),
        "sg_raw_excess_ann_vol": sg_realized_excess_vol,
        "sg_target_excess_ann_vol": sg_scaled_excess_returns.std() * np.sqrt(252),
        "dbmf_raw_excess_ann_vol": dbmf_realized_excess_vol,
        "dbmf_target_excess_ann_vol": dbmf_scaled_excess_returns.std() * np.sqrt(252),
        "proxy_total_ann_vol": mf_returns.std() * np.sqrt(252),
    }
)

## Rolling managed-futures volatility

Vol targeting is applied to returns above cash. This chart compares the raw one-year realized excess volatility of DBMF and SG Trend against their configurable target-vol assumptions.

In [ ]:
rolling_excess_vol = pd.DataFrame(
    {
        "DBMF excess vol": dbmf_excess_returns.rolling(252).std() * np.sqrt(252),
        f"{MF_INDEX_COLUMN} excess vol": sg_excess_returns.rolling(252).std() * np.sqrt(252),
    }
)
ax = rolling_excess_vol.plot(title="Rolling 1Y managed-futures excess volatility", figsize=(11, 5))
ax.axhline(DBMF_EXCESS_TARGET_VOL, color="tab:blue", linestyle="--", linewidth=1, alpha=0.7, label="DBMF target vol")
ax.axhline(SG_EXCESS_TARGET_VOL, color="tab:orange", linestyle="--", linewidth=1, alpha=0.7, label="SG target vol")
ax.set_ylabel("Annualized volatility")
ax.legend(loc="best")

## Portfolio construction

Managed-futures sizing is funded from cash first, but cash is capped at the configured floor. Once the cash leg would breach that floor, the remainder is funded from `SECONDARY_FUNDING_WEIGHTS`.

In [ ]:
def funded_weights(mf_weight: float) -> pd.Series:
    weights = pd.Series(CORE_WEIGHTS, dtype=float)
    primary = pd.Series(PRIMARY_FUNDING_WEIGHTS, dtype=float)
    secondary = pd.Series(SECONDARY_FUNDING_WEIGHTS, dtype=float)

    available_cash_funding = max(0.0, weights[CASH_TICKER] - CASH_WEIGHT_FLOOR)
    primary_amount = min(mf_weight, available_cash_funding)
    secondary_amount = mf_weight - primary_amount

    weights = weights.sub(primary * primary_amount, fill_value=0.0)
    weights = weights.sub(secondary * secondary_amount, fill_value=0.0)
    weights["MF"] = mf_weight
    return weights


asset_returns = daily_returns[[EQUITY_TICKER, BOND_TICKER, COMMODITY_TICKER, CASH_TICKER]].join(mf_returns.rename("MF"), how="inner")
portfolio_returns = {}
candidate_weights = {}

for mf_weight in DBMF_WEIGHTS:
    weights = funded_weights(mf_weight)
    name = f"{mf_weight:.1%} MF"
    candidate_weights[name] = weights
    portfolio_returns[name] = fixed_weight_returns(asset_returns, weights)

portfolio_returns = pd.DataFrame(portfolio_returns).dropna(how="any")
candidate_weights = pd.DataFrame(candidate_weights).T
candidate_weights

In [ ]:
portfolio_returns.tail()

## Decision table

Pick the highest managed-futures allocation where monthly CVaR and max drawdown improve, while MF risk contribution remains below the policy cap. The stress columns use monthly returns, because daily noise can hide the portfolio-level role.

In [ ]:
def monthly_returns(returns: pd.Series | pd.DataFrame) -> pd.Series | pd.DataFrame:
    return (1 + returns).resample("ME").prod() - 1


monthly_assets = monthly_returns(asset_returns)
monthly_portfolios = monthly_returns(portfolio_returns)

equity_crash = monthly_assets[EQUITY_TICKER] < -0.05
bond_shock = monthly_assets[BOND_TICKER] < -0.03
equity_bond_down = (monthly_assets[EQUITY_TICKER] < 0) & (monthly_assets[BOND_TICKER] < 0)
combined_core_shock = (monthly_assets[EQUITY_TICKER] + monthly_assets[BOND_TICKER]).rank(pct=True) <= 0.10
equity_bond_selloff = equity_bond_down & combined_core_shock

stress_counts = pd.Series(
    {
        "equity_crash_months": int(equity_crash.sum()),
        "bond_shock_months": int(bond_shock.sum()),
        "equity_bond_down_months": int(equity_bond_down.sum()),
        "equity_bond_selloff_months": int(equity_bond_selloff.sum()),
    }
)
stress_counts

In [ ]:
def stress_mean(returns: pd.DataFrame, mask: pd.Series) -> pd.Series:
    mask = mask.reindex(returns.index).fillna(False)
    if int(mask.sum()) < MIN_STRESS_MONTHS:
        return pd.Series(np.nan, index=returns.columns)
    return returns.loc[mask].mean()


def mf_risk_contribution_pct(weights: pd.Series, returns: pd.DataFrame) -> float:
    cov = cov_matrix(returns.reindex(columns=weights.index).dropna(), periods=252)
    rc = risk_contributions(weights, cov)
    total = rc.sum()
    return float(rc.get("MF", 0.0) / total) if total != 0 else np.nan


stats = pd.DataFrame({name: pd.Series(summary(series)) for name, series in portfolio_returns.items()}).T
stats["MonthlyCVaR5"] = monthly_portfolios.apply(cvar, q=MONTHLY_CVAR_Q)
stats["Worst1M"] = monthly_portfolios.min()
stats["EquityCrashAvg"] = stress_mean(monthly_portfolios, equity_crash)
stats["BondShockAvg"] = stress_mean(monthly_portfolios, bond_shock)
stats["EquityBondDownAvg"] = stress_mean(monthly_portfolios, equity_bond_down)
stats["WorstEquityBondSelloffAvg"] = stress_mean(monthly_portfolios, equity_bond_selloff)
stats["CorrToEquity"] = portfolio_returns.corrwith(asset_returns[EQUITY_TICKER])
stats["CorrTo10YProxy"] = portfolio_returns.corrwith(asset_returns[BOND_TICKER])
stats["MFRiskContribution"] = [
    mf_risk_contribution_pct(candidate_weights.loc[name], asset_returns) for name in stats.index
]
stats["FinalWealth"] = cumulative_returns(portfolio_returns).iloc[-1]

decision_cols = [
    "CAGR",
    "AnnVol",
    "Sharpe",
    "MaxDD",
    "MonthlyCVaR5",
    "WorstEquityBondSelloffAvg",
    "CorrToEquity",
    "CorrTo10YProxy",
    "MFRiskContribution",
    "FinalWealth",
]
stats[decision_cols]

In [ ]:
base = stats.loc["0.0% MF", decision_cols]
delta_vs_base = stats[decision_cols].sub(base, axis=1)
delta_vs_base

In [ ]:
eligible = stats[
    (stats["MonthlyCVaR5"] > stats.loc["0.0% MF", "MonthlyCVaR5"])
    & (stats["MaxDD"] > stats.loc["0.0% MF", "MaxDD"])
    & (stats["MFRiskContribution"] <= 0.15)
]

pd.Series(
    {
        "rule": "highest weight where CVaR and MaxDD improve and MF risk contribution <= 15%",
        "candidate": eligible.index[-1] if not eligible.empty else "no allocation passes all gates",
        "source": source_label,
    }
)

## Conditional managed-futures behavior

These rows answer the bad-times-beta question directly. If the managed-futures proxy does not help during equity-bond down months, keep sizing closer to 5-10% even if full-sample correlation looks low.

In [ ]:
mf_monthly = monthly_assets["MF"]
conditional_mf = pd.Series(
    {
        "all_months": mf_monthly.mean(),
        "equity_crash": mf_monthly.loc[equity_crash].mean(),
        "bond_shock": mf_monthly.loc[bond_shock].mean(),
        "equity_bond_down": mf_monthly.loc[equity_bond_down].mean(),
        "worst_equity_bond_selloff": mf_monthly.loc[equity_bond_selloff].mean(),
    }
)
conditional_mf

In [ ]:
rolling_corr = monthly_assets["MF"].rolling(ROLLING_WINDOW_MONTHS).corr(monthly_assets[EQUITY_TICKER])
rolling_corr.to_frame("MF rolling corr to equity").join(
    monthly_assets["MF"].rolling(ROLLING_WINDOW_MONTHS).corr(monthly_assets[BOND_TICKER]).rename("MF rolling corr to 10Y proxy")
).plot(title=f"Rolling {ROLLING_WINDOW_MONTHS}M managed-futures correlations")

## Charts

In [ ]:
equity_curve(portfolio_returns)

In [ ]:
drawdown(portfolio_returns).plot(title="Drawdowns by managed-futures allocation")

In [ ]:
best_by_rule = eligible.index[-1] if not eligible.empty else stats["Sharpe"].idxmax()
drawdown_chart(portfolio_returns[best_by_rule])

In [ ]:
heatmap_monthly(portfolio_returns[best_by_rule])

## Crisis period plots

Each chart rebases selected candidate portfolios to 1.0 at the start of the stress window. The focus is baseline vs practical managed-futures allocations, not every grid point.

In [ ]:
CRISIS_COLUMNS = ["0.0% MF", "7.5% MF", "10.0% MF", "15.0% MF", best_by_rule]
CRISIS_COLUMNS = list(dict.fromkeys(col for col in CRISIS_COLUMNS if col in portfolio_returns.columns))


def crisis_period_plot(name: str, start: str, end: str):
    window = portfolio_returns.loc[start:end, CRISIS_COLUMNS]
    if window.empty:
        return pd.Series({"name": name, "start": start, "end": end, "status": "no overlapping data"})

    wealth = cumulative_returns(window)
    ax = wealth.plot(title=f"{name}: rebased wealth", figsize=(11, 5))
    ax.axhline(1.0, color="black", linewidth=0.8, alpha=0.5)
    ax.set_ylabel("Wealth, start = 1")
    ax.legend(loc="best")
    return ax

In [ ]:
crisis_period_plot("Global financial crisis", "2007-10-09", "2009-03-09")

In [ ]:
crisis_period_plot("Euro crisis / US downgrade", "2011-07-01", "2011-10-03")

In [ ]:
crisis_period_plot("Q4 2018 tightening scare", "2018-09-20", "2018-12-24")

In [ ]:
crisis_period_plot("COVID crash", "2020-02-19", "2020-03-23")

In [ ]:
crisis_period_plot("2022 inflation / rate shock", "2022-01-03", "2022-10-14")

In [ ]:
crisis_period_plot("Liberation Day tariffs", "2025-04-02", "2025-05-15")

In [ ]:
crisis_period_plot("Iran Conflict", "2026-02-27", "2026-05-15")

## Policy interpretation

Use this notebook to maintain the policy range:

- Strategic target: 10% managed futures.
- Normal range: 5-15%.
- Move toward 15% only if the stress rows improve and MF risk contribution stays below 15%.
- Avoid 20%+ unless trend following is intentionally promoted to a core risk sleeve rather than a diversifier.

Before changing actual sizing, check DBMF's current holdings / inferred positions so the sleeve is not silently doubling existing duration, commodity, or equity exposure.